In [3]:
# ============================================================
# QCTrojan-Bench: Quantum Circuit Trojan Detection Benchmark
# ============================================================
# Notebook:  S1_benign_circuits.ipynb
# Purpose:   Generate 3,000 benign quantum circuits across
#            five algorithm families (600 per family)
# Authors:   Zeeshan Ajmal
#            University of Oulu, Finland
# Version:   QCTrojan-Bench v1.0
# License:   CC BY 4.0
# ============================================================
#
# CELL 1 — Imports and Configuration
#
# All paths, seeds, and constants are defined here.
# No other cell hardcodes any value.
# Run this cell first before any other cell.
# ============================================================

import os
import json
import random
from pathlib import Path
from datetime import datetime, timezone

from qiskit import QuantumCircuit, qpy
from qiskit.synthesis.qft import synth_qft_full
from qiskit.circuit.library import n_local

# ── Versioning ───────────────────────────────────────────────
QISKIT_VERSION    = "2.3.1"
GENERATOR_VERSION = "QCTrojan-Bench-v1.0"
DATASET_VERSION   = "v1"

# ── Paths (auto-detected — no hardcoding needed) ─────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATASET_ROOT = PROJECT_ROOT / "dataset"

BENIGN_DIR   = DATASET_ROOT / "circuits" / "benign"
TAMPERED_DIR = DATASET_ROOT / "circuits" / "tampered"
FEATURES_DIR = DATASET_ROOT / "features"

# ── Dataset constants ─────────────────────────────────────────
NUM_CIRCUITS = 600

# ── Per-family configuration ──────────────────────────────────
FAMILY_CONFIG = {
    "deutsch_jozsa": {
        "base_seed": 1000,
        "qubit_min": 3,
        "qubit_max": 8,
    },
    "grover": {
        "base_seed": 2000,
        "qubit_min": 4,
        "qubit_max": 12,
    },
    "qaoa": {
        "base_seed": 3000,
        "qubit_min": 6,
        "qubit_max": 14,
    },
    "vqc": {
        "base_seed": 4000,
        "qubit_min": 4,
        "qubit_max": 12,
    },
    "qft": {
        "base_seed": 5000,
        "qubit_min": 4,
        "qubit_max": 12,
    },
}

# ── Create output directories ─────────────────────────────────
for family in FAMILY_CONFIG:
    (BENIGN_DIR / family / DATASET_VERSION / "circuits").mkdir(
        parents=True, exist_ok=True)
    (BENIGN_DIR / family / DATASET_VERSION / "metadata").mkdir(
        parents=True, exist_ok=True)

# ── Sanity check ──────────────────────────────────────────────
print("=" * 55)
print("QCTrojan-Bench — S1 Benign Circuit Generation")
print("=" * 55)
print(f"  Generator version : {GENERATOR_VERSION}")
print(f"  Qiskit version    : {QISKIT_VERSION}")
print(f"  Project root      : {PROJECT_ROOT}")
print(f"  Output directory  : {BENIGN_DIR}")
print(f"  Circuits/family   : {NUM_CIRCUITS}")
print(f"  Families          : {list(FAMILY_CONFIG.keys())}")
print(f"  Total expected    : {NUM_CIRCUITS * len(FAMILY_CONFIG):,}")
print("=" * 55)
print("Cell 1 complete. Directories created. Ready for Cell 2.")

QCTrojan-Bench — S1 Benign Circuit Generation
  Generator version : QCTrojan-Bench-v1.0
  Qiskit version    : 2.3.1
  Project root      : C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp
  Output directory  : C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\dataset\circuits\benign
  Circuits/family   : 600
  Families          : ['deutsch_jozsa', 'grover', 'qaoa', 'vqc', 'qft']
  Total expected    : 3,000
Cell 1 complete. Directories created. Ready for Cell 2.


In [ ]:
#---
#
# **Three things to note before you run this:**
# 
# First, the notebook must live at `project_root/notebooks/S1_benign_circuits.ipynb`. The path resolution goes one level up from the notebook location to find the project root. So your structure should be:
# ```
# quantum_circuits_exp/
#   notebooks/
#     S1_benign_circuits.ipynb   ← notebook lives here
#   dataset/
#     circuits/
#       benign/                  ← auto-created by Cell 1
#       tampered/
#     features/
# ```
# 
# Second, if you open Jupyter from a different working directory than the notebook's location, `Path().resolve()` returns the wrong path. Always open Jupyter from `quantum_circuits_exp/` and navigate to the notebook from there.
# 
# Third, the `.gitignore` needs updating before you push. The QPY circuit files are binary and large — they should not go into GitHub. Add these lines to your `.gitignore`:
# ```
# dataset/circuits/benign/**/*.qpy
# dataset/circuits/tampered/**/*.qpy
# dataset/features/*.csv

In [4]:
# ============================================================
# CELL 2 — Generate Deutsch-Jozsa Benign Circuits
# ============================================================
#
# Generates 600 benign Deutsch-Jozsa circuits.
#
# Design:
#   - Two oracle types: constant (do nothing) and balanced
#     (CNOT chain from each input qubit to last qubit)
#   - Qubit count varied: 3 to 8
#   - Standard structure: H layer → oracle → H layer
#   - No measurements (pure unitary circuits)
#   - Fully deterministic given base_seed
#
# Output:
#   dataset/circuits/benign/deutsch_jozsa/v1/circuits/*.qpy
#   dataset/circuits/benign/deutsch_jozsa/v1/metadata/*.json
# ============================================================

FAMILY     = "deutsch_jozsa"
CFG        = FAMILY_CONFIG[FAMILY]
BASE_SEED  = CFG["base_seed"]
QUBIT_MIN  = CFG["qubit_min"]
QUBIT_MAX  = CFG["qubit_max"]

CIRCUIT_DIR  = BENIGN_DIR / FAMILY / DATASET_VERSION / "circuits"
METADATA_DIR = BENIGN_DIR / FAMILY / DATASET_VERSION / "metadata"

# ── Circuit builder ───────────────────────────────────────────

def build_dj_circuit(n_qubits: int, oracle_type: str) -> QuantumCircuit:
    """
    Build a Deutsch-Jozsa circuit without measurement.

    Args:
        n_qubits:    total qubit count (input qubits = n_qubits - 1,
                     ancilla = last qubit)
        oracle_type: 'constant' (identity) or 'balanced' (CNOT chain)

    Returns:
        QuantumCircuit
    """
    qc = QuantumCircuit(n_qubits)

    # Standard initialisation: |0...01> → H on all qubits
    qc.x(n_qubits - 1)          # ancilla to |1>
    qc.h(range(n_qubits))       # superposition

    # Oracle
    if oracle_type == "constant":
        pass                     # constant-0: identity, no gates
    elif oracle_type == "balanced":
        for i in range(n_qubits - 1):
            qc.cx(i, n_qubits - 1)
    else:
        raise ValueError(f"Unknown oracle_type: {oracle_type}")

    # Final H layer on input qubits only
    qc.h(range(n_qubits - 1))

    return qc


# ── Generation loop ───────────────────────────────────────────

rng        = random.Random(BASE_SEED)
oracle_choices = ["constant", "balanced"]
generated  = 0
skipped    = 0

for idx in range(NUM_CIRCUITS):
    seed_i      = BASE_SEED + idx
    n_qubits    = rng.randint(QUBIT_MIN, QUBIT_MAX)
    oracle_type = rng.choice(oracle_choices)

    sample_id = f"benign_{FAMILY}_{DATASET_VERSION}_{idx:06d}"

    # Skip if already exists (safe re-run)
    qpy_path = CIRCUIT_DIR / f"{sample_id}.qpy"
    if qpy_path.exists():
        skipped += 1
        continue

    # Build circuit
    qc = build_dj_circuit(n_qubits, oracle_type)

    # Save QPY
    with open(qpy_path, "wb") as f:
        qpy.dump(qc, f)

    # Save metadata
    metadata = {
        "sample_id":          sample_id,
        "label":              "benign",
        "trojan_type":        "benign",
        "trojan_severity":    None,
        "algorithm_family":   FAMILY,
        "algorithm_version":  DATASET_VERSION,
        "parent_sample_id":   None,
        "n_qubits":           n_qubits,
        "oracle_type":        oracle_type,
        "generation_index":   idx,
        "generation_seed":    seed_i,
        "generator_version":  GENERATOR_VERSION,
        "qiskit_version":     QISKIT_VERSION,
        "created":            datetime.now(timezone.utc).isoformat(),
    }

    with open(METADATA_DIR / f"{sample_id}.json", "w") as f:
        json.dump(metadata, f, indent=2)

    generated += 1

# ── Validation ────────────────────────────────────────────────

# Count saved circuits and verify no measurements
qpy_files = sorted(CIRCUIT_DIR.glob("*.qpy"))
assert len(qpy_files) == NUM_CIRCUITS, \
    f"Expected {NUM_CIRCUITS} circuits, found {len(qpy_files)}"

for qpy_file in qpy_files:
    with open(qpy_file, "rb") as f:
        qc_check = qpy.load(f)[0]
    assert qc_check.num_clbits == 0, \
        f"Measurement found in {qpy_file.name}"

print(f"Deutsch-Jozsa generation complete.")
print(f"  Generated : {generated}")
print(f"  Skipped   : {skipped} (already existed)")
print(f"  Validated : {len(qpy_files)} circuits — no measurements ✓")
print(f"  Saved to  : {CIRCUIT_DIR}")

Deutsch-Jozsa generation complete.
  Generated : 600
  Skipped   : 0 (already existed)
  Validated : 600 circuits — no measurements ✓
  Saved to  : C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\dataset\circuits\benign\deutsch_jozsa\v1\circuits


In [5]:
# ============================================================
# CELL 3 — Generate Grover Search Benign Circuits
# ============================================================
#
# Generates 600 benign Grover search circuits.
#
# Design:
#   - Three oracle types:
#       * bitstring: phase flip on one marked basis state
#       * cnf: phase flip on states satisfying a conjunction
#       * parity: phase flip based on qubit parity
#   - Two diffusion operator implementations:
#       * standard_mcz: H/X + multi-controlled Z + X/H
#       * mcp_phase: multi-controlled phase variant
#   - Iterations: 1 to floor(pi/4 * sqrt(2^n))  capped at 3
#   - Qubit count varied: 4 to 12
#   - No measurements
#   - Fully deterministic given base_seed
#
# Output:
#   dataset/circuits/benign/grover/v1/circuits/*.qpy
#   dataset/circuits/benign/grover/v1/metadata/*.json
# ============================================================

import math
from qiskit.circuit.library import MCXGate

FAMILY     = "grover"
CFG        = FAMILY_CONFIG[FAMILY]
BASE_SEED  = CFG["base_seed"]
QUBIT_MIN  = CFG["qubit_min"]
QUBIT_MAX  = CFG["qubit_max"]

CIRCUIT_DIR  = BENIGN_DIR / FAMILY / DATASET_VERSION / "circuits"
METADATA_DIR = BENIGN_DIR / FAMILY / DATASET_VERSION / "metadata"

# ── Building blocks ───────────────────────────────────────────

def _mcz(qc: QuantumCircuit, controls: list, target: int) -> None:
    """Multi-controlled Z via H — MCX — H on target."""
    qc.h(target)
    qc.append(MCXGate(len(controls)), controls + [target])
    qc.h(target)


def oracle_bitstring(qc: QuantumCircuit, marked: str) -> None:
    """Phase flip exactly one basis state |marked>."""
    n = qc.num_qubits
    for i, b in enumerate(marked):
        if b == "0":
            qc.x(i)
    _mcz(qc, list(range(n - 1)), n - 1)
    for i, b in enumerate(marked):
        if b == "0":
            qc.x(i)


def oracle_cnf(qc: QuantumCircuit, literals: dict) -> None:
    """Phase flip states satisfying a conjunction of literals."""
    n = qc.num_qubits
    controls = sorted(literals.keys())
    target = next((i for i in range(n) if i not in controls), n - 1)
    controls = [c for c in controls if c != target]
    if not controls:
        return
    for q in controls:
        if literals[q] == 0:
            qc.x(q)
    _mcz(qc, controls, target)
    for q in controls:
        if literals[q] == 0:
            qc.x(q)


def oracle_parity(qc: QuantumCircuit, subset: list) -> None:
    """Phase flip based on parity of a qubit subset."""
    if len(subset) < 2:
        return
    for q in subset:
        qc.z(q)


def diffusion_standard(qc: QuantumCircuit) -> None:
    """Standard Grover diffusion: H/X + MCZ + X/H."""
    n = qc.num_qubits
    qc.h(range(n))
    qc.x(range(n))
    _mcz(qc, list(range(n - 1)), n - 1)
    qc.x(range(n))
    qc.h(range(n))


def diffusion_mcp(qc: QuantumCircuit) -> None:
    """Diffusion via multi-controlled phase — structural variant."""
    n = qc.num_qubits
    qc.h(range(n))
    qc.x(range(n))
    qc.h(n - 1)
    qc.append(MCXGate(n - 1), list(range(n - 1)) + [n - 1])
    qc.h(n - 1)
    qc.x(range(n))
    qc.h(range(n))


def build_grover_circuit(
    n_qubits: int,
    oracle_type: str,
    diffusion_impl: str,
    num_iterations: int,
    rng: random.Random,
) -> tuple:
    """
    Build a Grover circuit without measurement.

    Returns:
        (QuantumCircuit, oracle_params dict)
    """
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))   # uniform superposition

    oracle_params = {}

    for _ in range(num_iterations):

        # ── Oracle ──────────────────────────────────────────
        if oracle_type == "bitstring":
            marked = "".join(rng.choice("01") for _ in range(n_qubits))
            oracle_bitstring(qc, marked)
            oracle_params["marked_state"] = marked

        elif oracle_type == "cnf":
            num_literals = rng.randint(2, min(5, n_qubits))
            qubits = rng.sample(range(n_qubits), num_literals)
            literals = {q: rng.randint(0, 1) for q in qubits}
            oracle_cnf(qc, literals)
            oracle_params["cnf_literals"] = literals

        elif oracle_type == "parity":
            size = rng.randint(2, max(2, n_qubits // 2))
            subset = rng.sample(range(n_qubits), size)
            oracle_parity(qc, subset)
            oracle_params["parity_subset"] = subset

        # ── Diffusion ────────────────────────────────────────
        if diffusion_impl == "standard_mcz":
            diffusion_standard(qc)
        else:
            diffusion_mcp(qc)

    return qc, oracle_params


# ── Generation loop ───────────────────────────────────────────

rng = random.Random(BASE_SEED)

ORACLE_TYPES    = ["bitstring", "cnf", "parity"]
DIFFUSION_IMPLS = ["standard_mcz", "mcp_phase"]

generated = 0
skipped   = 0

for idx in range(NUM_CIRCUITS):
    seed_i         = BASE_SEED + idx
    local_rng      = random.Random(seed_i)

    n_qubits       = local_rng.randint(QUBIT_MIN, QUBIT_MAX)
    oracle_type    = local_rng.choice(ORACLE_TYPES)
    diffusion_impl = local_rng.choice(DIFFUSION_IMPLS)

    # Grover optimal iterations: floor(pi/4 * sqrt(N)), capped at 3
    max_iters      = max(1, min(3, int(math.floor(
                         (math.pi / 4) * math.sqrt(2 ** n_qubits)))))
    num_iterations = local_rng.randint(1, max_iters)

    sample_id = f"benign_{FAMILY}_{DATASET_VERSION}_{idx:06d}"

    qpy_path = CIRCUIT_DIR / f"{sample_id}.qpy"
    if qpy_path.exists():
        skipped += 1
        continue

    qc, oracle_params = build_grover_circuit(
        n_qubits, oracle_type, diffusion_impl, num_iterations, local_rng
    )

    # Save QPY
    with open(qpy_path, "wb") as f:
        qpy.dump(qc, f)

    # Save metadata
    metadata = {
        "sample_id":          sample_id,
        "label":              "benign",
        "trojan_type":        "benign",
        "trojan_severity":    None,
        "algorithm_family":   FAMILY,
        "algorithm_version":  DATASET_VERSION,
        "parent_sample_id":   None,
        "n_qubits":           n_qubits,
        "oracle_type":        oracle_type,
        "diffusion_impl":     diffusion_impl,
        "num_iterations":     num_iterations,
        "oracle_params":      oracle_params,
        "generation_index":   idx,
        "generation_seed":    seed_i,
        "generator_version":  GENERATOR_VERSION,
        "qiskit_version":     QISKIT_VERSION,
        "created":            datetime.now(timezone.utc).isoformat(),
    }

    with open(METADATA_DIR / f"{sample_id}.json", "w") as f:
        json.dump(metadata, f, indent=2)

    generated += 1

# ── Validation ────────────────────────────────────────────────

qpy_files = sorted(CIRCUIT_DIR.glob("*.qpy"))
assert len(qpy_files) == NUM_CIRCUITS, \
    f"Expected {NUM_CIRCUITS} circuits, found {len(qpy_files)}"

for qpy_file in qpy_files:
    with open(qpy_file, "rb") as f:
        qc_check = qpy.load(f)[0]
    assert qc_check.num_clbits == 0, \
        f"Measurement found in {qpy_file.name}"

print(f"Grover generation complete.")
print(f"  Generated : {generated}")
print(f"  Skipped   : {skipped} (already existed)")
print(f"  Validated : {len(qpy_files)} circuits — no measurements ✓")
print(f"  Saved to  : {CIRCUIT_DIR}")

Grover generation complete.
  Generated : 600
  Skipped   : 0 (already existed)
  Validated : 600 circuits — no measurements ✓
  Saved to  : C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\dataset\circuits\benign\grover\v1\circuits


In [8]:
pip install networkx

  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# ============================================================
# CELL 4 — Generate QAOA Benign Circuits
# ============================================================
#
# Generates 600 benign QAOA (MaxCut) circuits.
#
# Design:
#   - Problem: MaxCut on Erdos-Renyi random graphs
#   - Cost unitary: RZZ(2*gamma) on each edge
#   - Mixer unitary: RX(2*beta) on each qubit
#   - Initial state: |+>^n (H on all qubits)
#   - p layers: 1, 2, or 3
#   - Parameters: symbolic (not bound — pure ansatz structure)
#   - Edge probability: 0.3 to 0.6
#   - Qubit count: 6 to 14
#   - No measurements
#   - Fully deterministic given base_seed
#
# Output:
#   dataset/circuits/benign/qaoa/v1/circuits/*.qpy
#   dataset/circuits/benign/qaoa/v1/metadata/*.json
# ============================================================

import networkx as nx
from qiskit.circuit import ParameterVector

FAMILY     = "qaoa"
CFG        = FAMILY_CONFIG[FAMILY]
BASE_SEED  = CFG["base_seed"]
QUBIT_MIN  = CFG["qubit_min"]
QUBIT_MAX  = CFG["qubit_max"]

CIRCUIT_DIR  = BENIGN_DIR / FAMILY / DATASET_VERSION / "circuits"
METADATA_DIR = BENIGN_DIR / FAMILY / DATASET_VERSION / "metadata"

# ── Circuit builder ───────────────────────────────────────────

def build_qaoa_circuit(
    n_qubits: int,
    p_layers: int,
    edges: list,
) -> QuantumCircuit:
    """
    Build a parameterized QAOA MaxCut circuit without measurement.

    Parameters are symbolic (not bound). This preserves the
    structural ansatz for feature extraction without requiring
    classical optimization.

    Args:
        n_qubits: number of qubits
        p_layers: number of QAOA layers
        edges:    list of (u, v) tuples defining the MaxCut graph

    Returns:
        QuantumCircuit with 2*p_layers symbolic parameters
    """
    gammas = ParameterVector("gamma", p_layers)
    betas  = ParameterVector("beta",  p_layers)

    qc = QuantumCircuit(n_qubits)

    # Initial state: uniform superposition
    qc.h(range(n_qubits))

    for layer in range(p_layers):
        g = gammas[layer]
        b = betas[layer]

        # Cost unitary: exp(-i gamma Z_i Z_j) per edge
        for (u, v) in edges:
            qc.rzz(2 * g, u, v)

        # Mixer unitary: exp(-i beta X_i) per qubit
        for q in range(n_qubits):
            qc.rx(2 * b, q)

    return qc


# ── Generation loop ───────────────────────────────────────────

P_CHOICES      = [1, 2, 3]
EDGE_P_MIN     = 0.3
EDGE_P_MAX     = 0.6

generated = 0
skipped   = 0

for idx in range(NUM_CIRCUITS):
    seed_i     = BASE_SEED + idx
    local_rng  = random.Random(seed_i)

    n_qubits   = local_rng.randint(QUBIT_MIN, QUBIT_MAX)
    p_layers   = local_rng.choice(P_CHOICES)
    edge_prob  = local_rng.uniform(EDGE_P_MIN, EDGE_P_MAX)

    # Deterministic graph — separate seed for auditability
    graph_seed = seed_i + 100_000
    G = nx.erdos_renyi_graph(
        n=n_qubits, p=edge_prob, seed=graph_seed, directed=False
    )
    edges = sorted(
        (min(u, v), max(u, v)) for (u, v) in G.edges()
    )

    sample_id = f"benign_{FAMILY}_{DATASET_VERSION}_{idx:06d}"

    qpy_path = CIRCUIT_DIR / f"{sample_id}.qpy"
    if qpy_path.exists():
        skipped += 1
        continue

    qc = build_qaoa_circuit(n_qubits, p_layers, edges)

    # Save QPY
    with open(qpy_path, "wb") as f:
        qpy.dump(qc, f)

    # Save metadata
    metadata = {
        "sample_id":          sample_id,
        "label":              "benign",
        "trojan_type":        "benign",
        "trojan_severity":    None,
        "algorithm_family":   FAMILY,
        "algorithm_version":  DATASET_VERSION,
        "parent_sample_id":   None,
        "n_qubits":           n_qubits,
        "p_layers":           p_layers,
        "num_edges":          len(edges),
        "edges":              edges,
        "edge_probability":   round(edge_prob, 6),
        "graph_seed":         graph_seed,
        "num_parameters":     2 * p_layers,
        "generation_index":   idx,
        "generation_seed":    seed_i,
        "generator_version":  GENERATOR_VERSION,
        "qiskit_version":     QISKIT_VERSION,
        "created":            datetime.now(timezone.utc).isoformat(),
    }

    with open(METADATA_DIR / f"{sample_id}.json", "w") as f:
        json.dump(metadata, f, indent=2)

    generated += 1

# ── Validation ────────────────────────────────────────────────

qpy_files = sorted(CIRCUIT_DIR.glob("*.qpy"))
assert len(qpy_files) == NUM_CIRCUITS, \
    f"Expected {NUM_CIRCUITS} circuits, found {len(qpy_files)}"

for qpy_file in qpy_files:
    with open(qpy_file, "rb") as f:
        qc_check = qpy.load(f)[0]
    assert qc_check.num_clbits == 0, \
        f"Measurement found in {qpy_file.name}"

print(f"QAOA generation complete.")
print(f"  Generated : {generated}")
print(f"  Skipped   : {skipped} (already existed)")
print(f"  Validated : {len(qpy_files)} circuits — no measurements ✓")
print(f"  Saved to  : {CIRCUIT_DIR}")

QAOA generation complete.
  Generated : 600
  Skipped   : 0 (already existed)
  Validated : 600 circuits — no measurements ✓
  Saved to  : C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\dataset\circuits\benign\qaoa\v1\circuits


In [10]:
# ============================================================
# CELL 5 — Generate VQC (TwoLocal) Benign Circuits
# ============================================================
#
# Generates 600 benign Variational Quantum Circuit (VQC)
# circuits using the TwoLocal ansatz structure.
#
# Design:
#   - Ansatz: TwoLocal (hardware-efficient, problem-agnostic)
#   - Rotation blocks: [ry] or [rx, ry]
#   - Entanglement: CX gates, linear or full pattern
#   - Reps (ansatz depth): 1 to 4
#   - Parameters: symbolic (not bound)
#   - Qubit count: 4 to 12
#   - No measurements
#   - Fully deterministic given base_seed
#
# Output:
#   dataset/circuits/benign/vqc/v1/circuits/*.qpy
#   dataset/circuits/benign/vqc/v1/metadata/*.json
# ============================================================

FAMILY     = "vqc"
CFG        = FAMILY_CONFIG[FAMILY]
BASE_SEED  = CFG["base_seed"]
QUBIT_MIN  = CFG["qubit_min"]
QUBIT_MAX  = CFG["qubit_max"]

CIRCUIT_DIR  = BENIGN_DIR / FAMILY / DATASET_VERSION / "circuits"
METADATA_DIR = BENIGN_DIR / FAMILY / DATASET_VERSION / "metadata"

# ── Circuit builder ───────────────────────────────────────────

def build_vqc_circuit(
    n_qubits: int,
    reps: int,
    rotation_blocks: list,
    entanglement: str,
) -> QuantumCircuit:
    """
    Build a parameterized TwoLocal VQC circuit without measurement.

    Parameters are symbolic (not bound). The circuit structure
    captures hardware-efficient ansatz patterns used in
    variational quantum algorithms.

    Args:
        n_qubits:        number of qubits
        reps:            number of alternating rotation+entanglement layers
        rotation_blocks: list of rotation gate names e.g. ['ry'] or ['rx','ry']
        entanglement:    'linear' or 'full' CX entanglement pattern

    Returns:
        QuantumCircuit with symbolic parameters
    """
    qc = n_local(
        num_qubits=n_qubits,
        rotation_blocks=rotation_blocks,
        entanglement_blocks="cx",
        entanglement=entanglement,
        reps=reps,
        parameter_prefix="θ",
    )
    return qc


# ── Generation loop ───────────────────────────────────────────

REPS_CHOICES       = [1, 2, 3, 4]
ROTATION_CHOICES   = [["ry"], ["rx", "ry"]]
ENTANGLEMENT_CHOICES = ["linear", "full"]

generated = 0
skipped   = 0

for idx in range(NUM_CIRCUITS):
    seed_i    = BASE_SEED + idx
    local_rng = random.Random(seed_i)

    n_qubits        = local_rng.randint(QUBIT_MIN, QUBIT_MAX)
    reps            = local_rng.choice(REPS_CHOICES)
    rotation_blocks = local_rng.choice(ROTATION_CHOICES)
    entanglement    = local_rng.choice(ENTANGLEMENT_CHOICES)

    sample_id = f"benign_{FAMILY}_{DATASET_VERSION}_{idx:06d}"

    qpy_path = CIRCUIT_DIR / f"{sample_id}.qpy"
    if qpy_path.exists():
        skipped += 1
        continue

    qc = build_vqc_circuit(n_qubits, reps, rotation_blocks, entanglement)

    # Save QPY
    with open(qpy_path, "wb") as f:
        qpy.dump(qc, f)

    # Save metadata
    metadata = {
        "sample_id":          sample_id,
        "label":              "benign",
        "trojan_type":        "benign",
        "trojan_severity":    None,
        "algorithm_family":   FAMILY,
        "algorithm_version":  DATASET_VERSION,
        "parent_sample_id":   None,
        "n_qubits":           n_qubits,
        "reps":               reps,
        "rotation_blocks":    rotation_blocks,
        "entanglement":       entanglement,
        "num_parameters":     len(qc.parameters),
        "generation_index":   idx,
        "generation_seed":    seed_i,
        "generator_version":  GENERATOR_VERSION,
        "qiskit_version":     QISKIT_VERSION,
        "created":            datetime.now(timezone.utc).isoformat(),
    }

    with open(METADATA_DIR / f"{sample_id}.json", "w") as f:
        json.dump(metadata, f, indent=2)

    generated += 1

# ── Validation ────────────────────────────────────────────────

qpy_files = sorted(CIRCUIT_DIR.glob("*.qpy"))
assert len(qpy_files) == NUM_CIRCUITS, \
    f"Expected {NUM_CIRCUITS} circuits, found {len(qpy_files)}"

for qpy_file in qpy_files:
    with open(qpy_file, "rb") as f:
        qc_check = qpy.load(f)[0]
    assert qc_check.num_clbits == 0, \
        f"Measurement found in {qpy_file.name}"

print(f"VQC generation complete.")
print(f"  Generated : {generated}")
print(f"  Skipped   : {skipped} (already existed)")
print(f"  Validated : {len(qpy_files)} circuits — no measurements ✓")
print(f"  Saved to  : {CIRCUIT_DIR}")

VQC generation complete.
  Generated : 600
  Skipped   : 0 (already existed)
  Validated : 600 circuits — no measurements ✓
  Saved to  : C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\dataset\circuits\benign\vqc\v1\circuits


In [11]:
# ============================================================
# CELL 6 — Generate QFT Benign Circuits
# ============================================================
#
# Generates 600 benign Quantum Fourier Transform circuits.
#
# Design:
#   - Two directions: qft (forward) and iqft (inverse)
#   - Swap option: with or without final swap network
#   - Uses Qiskit's synth_qft_full for canonical construction
#   - Qubit count: 4 to 12
#   - No measurements
#   - Fully deterministic given base_seed
#
# Note on structural diversity:
#   QFT is deterministic given (n_qubits, direction, do_swaps).
#   Diversity comes from varying these three parameters across
#   the 600 circuits. This gives 2 directions x 2 swap options
#   x 9 qubit counts = 36 unique structural configurations,
#   each appearing multiple times across the 600 circuits.
#
# Output:
#   dataset/circuits/benign/qft/v1/circuits/*.qpy
#   dataset/circuits/benign/qft/v1/metadata/*.json
# ============================================================

FAMILY     = "qft"
CFG        = FAMILY_CONFIG[FAMILY]
BASE_SEED  = CFG["base_seed"]
QUBIT_MIN  = CFG["qubit_min"]
QUBIT_MAX  = CFG["qubit_max"]

CIRCUIT_DIR  = BENIGN_DIR / FAMILY / DATASET_VERSION / "circuits"
METADATA_DIR = BENIGN_DIR / FAMILY / DATASET_VERSION / "metadata"

# ── Circuit builder ───────────────────────────────────────────

def build_qft_circuit(
    n_qubits: int,
    direction: str,
    do_swaps: bool,
) -> QuantumCircuit:
    """
    Build a QFT or inverse QFT circuit without measurement.

    Uses Qiskit's synth_qft_full for the canonical decomposition
    into controlled phase rotations and Hadamard gates.

    Args:
        n_qubits:  number of qubits
        direction: 'qft' or 'iqft'
        do_swaps:  whether to include the final swap network

    Returns:
        QuantumCircuit
    """
    qc = synth_qft_full(
        num_qubits=n_qubits,
        do_swaps=do_swaps,
        inverse=(direction == "iqft"),
    )
    return qc


# ── Generation loop ───────────────────────────────────────────

DIRECTIONS = ["qft", "iqft"]
DO_SWAPS   = [True, False]

generated = 0
skipped   = 0

for idx in range(NUM_CIRCUITS):
    seed_i    = BASE_SEED + idx
    local_rng = random.Random(seed_i)

    n_qubits  = local_rng.randint(QUBIT_MIN, QUBIT_MAX)
    direction = local_rng.choice(DIRECTIONS)
    do_swaps  = local_rng.choice(DO_SWAPS)

    sample_id = f"benign_{FAMILY}_{DATASET_VERSION}_{idx:06d}"

    qpy_path = CIRCUIT_DIR / f"{sample_id}.qpy"
    if qpy_path.exists():
        skipped += 1
        continue

    qc = build_qft_circuit(n_qubits, direction, do_swaps)

    # Save QPY
    with open(qpy_path, "wb") as f:
        qpy.dump(qc, f)

    # Save metadata
    metadata = {
        "sample_id":         sample_id,
        "label":             "benign",
        "trojan_type":       "benign",
        "trojan_severity":   None,
        "algorithm_family":  FAMILY,
        "algorithm_version": DATASET_VERSION,
        "parent_sample_id":  None,
        "n_qubits":          n_qubits,
        "direction":         direction,
        "do_swaps":          do_swaps,
        "circuit_depth":     qc.depth(),
        "generation_index":  idx,
        "generation_seed":   seed_i,
        "generator_version": GENERATOR_VERSION,
        "qiskit_version":    QISKIT_VERSION,
        "created":           datetime.now(timezone.utc).isoformat(),
    }

    with open(METADATA_DIR / f"{sample_id}.json", "w") as f:
        json.dump(metadata, f, indent=2)

    generated += 1

# ── Validation ────────────────────────────────────────────────

qpy_files = sorted(CIRCUIT_DIR.glob("*.qpy"))
assert len(qpy_files) == NUM_CIRCUITS, \
    f"Expected {NUM_CIRCUITS} circuits, found {len(qpy_files)}"

for qpy_file in qpy_files:
    with open(qpy_file, "rb") as f:
        qc_check = qpy.load(f)[0]
    assert qc_check.num_clbits == 0, \
        f"Measurement found in {qpy_file.name}"

print(f"QFT generation complete.")
print(f"  Generated : {generated}")
print(f"  Skipped   : {skipped} (already existed)")
print(f"  Validated : {len(qpy_files)} circuits — no measurements ✓")
print(f"  Saved to  : {CIRCUIT_DIR}")

QFT generation complete.
  Generated : 600
  Skipped   : 0 (already existed)
  Validated : 600 circuits — no measurements ✓
  Saved to  : C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\dataset\circuits\benign\qft\v1\circuits


In [13]:
# ============================================================
# CELL 7 — Full Dataset Validation and Summary
# ============================================================
#
# Validates the complete S1 output across all five families.
# Run this cell after all five generation cells complete.
#
# Checks:
#   - Correct circuit count per family (600 each)
#   - No measurements in any circuit
#   - Every circuit has a matching metadata file
#   - Metadata schema is complete (no missing fields)
#   - No duplicate sample_ids across families
#   - Qubit counts within expected ranges
#   - Saves a dataset manifest (s1_manifest.json)
# ============================================================

from collections import defaultdict

REQUIRED_META_FIELDS = [
    "sample_id",
    "label",
    "trojan_type",
    "algorithm_family",
    "algorithm_version",
    "parent_sample_id",
    "n_qubits",
    "generation_index",
    "generation_seed",
    "generator_version",
    "qiskit_version",
    "created",
]

# ── Validation ────────────────────────────────────────────────

all_sample_ids  = []
family_summary  = {}
errors          = []

print("=" * 55)
print("S1 — Full Dataset Validation")
print("=" * 55)

for family, cfg in FAMILY_CONFIG.items():

    circuit_dir  = BENIGN_DIR / family / DATASET_VERSION / "circuits"
    metadata_dir = BENIGN_DIR / family / DATASET_VERSION / "metadata"

    qpy_files  = sorted(circuit_dir.glob("*.qpy"))
    json_files = sorted(metadata_dir.glob("*.json"))

    family_errors = []

    # ── Count check ──────────────────────────────────────────
    if len(qpy_files) != NUM_CIRCUITS:
        family_errors.append(
            f"Circuit count: expected {NUM_CIRCUITS}, "
            f"found {len(qpy_files)}"
        )

    if len(json_files) != NUM_CIRCUITS:
        family_errors.append(
            f"Metadata count: expected {NUM_CIRCUITS}, "
            f"found {len(json_files)}"
        )

    # ── Per-circuit checks ────────────────────────────────────
    qubit_counts = []

    for qpy_file in qpy_files:
        sample_id = qpy_file.stem
        all_sample_ids.append(sample_id)

        # Circuit loads and has no measurements
        try:
            with open(qpy_file, "rb") as f:
                qc = qpy.load(f)[0]
        except Exception as e:
            family_errors.append(f"QPY load failed {sample_id}: {e}")
            continue

        if qc.num_clbits > 0:
            family_errors.append(
                f"Measurement found in {sample_id}"
            )

        qubit_counts.append(qc.num_qubits)

        # Qubit range check
        if not (cfg["qubit_min"] <= qc.num_qubits <= cfg["qubit_max"]):
            family_errors.append(
                f"{sample_id}: n_qubits={qc.num_qubits} outside "
                f"[{cfg['qubit_min']}, {cfg['qubit_max']}]"
            )

        # Matching metadata exists
        meta_path = metadata_dir / f"{sample_id}.json"
        if not meta_path.exists():
            family_errors.append(
                f"Missing metadata for {sample_id}"
            )
            continue

        # Metadata schema check
        with open(meta_path) as f:
            meta = json.load(f)

        for field in REQUIRED_META_FIELDS:
            if field not in meta:
                family_errors.append(
                    f"{sample_id}: missing metadata field '{field}'"
                )

        # Label integrity
        if meta.get("label") != "benign":
            family_errors.append(
                f"{sample_id}: label is '{meta.get('label')}', "
                f"expected 'benign'"
            )

        if meta.get("trojan_type") != "benign":
            family_errors.append(
                f"{sample_id}: trojan_type is "
                f"'{meta.get('trojan_type')}', expected 'benign'"
            )

        if meta.get("parent_sample_id") is not None:
            family_errors.append(
                f"{sample_id}: parent_sample_id should be null "
                f"for benign circuits"
            )

    # ── Family summary ────────────────────────────────────────
    status = "✓ PASS" if not family_errors else "✗ FAIL"
    family_summary[family] = {
        "circuits":   len(qpy_files),
        "metadata":   len(json_files),
        "qubit_min":  min(qubit_counts) if qubit_counts else None,
        "qubit_max":  max(qubit_counts) if qubit_counts else None,
        "errors":     family_errors,
        "status":     status,
    }

    print(f"\n  {family} — {status}")
    print(f"    Circuits : {len(qpy_files)}")
    print(f"    Metadata : {len(json_files)}")
    if qubit_counts:
        print(f"    Qubits   : {min(qubit_counts)}–{max(qubit_counts)}")
    if family_errors:
        for err in family_errors[:5]:
            print(f"    ERROR: {err}")
        if len(family_errors) > 5:
            print(f"    ... and {len(family_errors) - 5} more errors")

    errors.extend(family_errors)

# ── Duplicate sample_id check ─────────────────────────────────
seen = set()
duplicates = []
for sid in all_sample_ids:
    if sid in seen:
        duplicates.append(sid)
    seen.add(sid)

if duplicates:
    errors.extend([f"Duplicate sample_id: {d}" for d in duplicates])
    print(f"\n  Duplicate IDs found: {duplicates}")

# ── Global summary ────────────────────────────────────────────
total_circuits = sum(
    v["circuits"] for v in family_summary.values()
)

print("\n" + "=" * 55)
print(f"  Total circuits : {total_circuits} / {NUM_CIRCUITS * len(FAMILY_CONFIG)} expected")
print(f"  Duplicate IDs  : {len(duplicates)}")
print(f"  Total errors   : {len(errors)}")

if errors:
    print("\n  ✗ VALIDATION FAILED — fix errors before proceeding")
else:
    print("\n  ✓ ALL CHECKS PASSED — S1 complete")

# ── Save manifest ─────────────────────────────────────────────
manifest = {
    "notebook":         "S1_benign_circuits.ipynb",
    "generator_version": GENERATOR_VERSION,
    "qiskit_version":   QISKIT_VERSION,
    "dataset_version":  DATASET_VERSION,
    "created":          datetime.now(timezone.utc).isoformat(),
    "total_circuits":   total_circuits,
    "families":         family_summary,
    "validation_passed": len(errors) == 0,
}

# Remove non-serialisable error lists if empty
for fam in manifest["families"]:
    manifest["families"][fam].pop("errors", None)

manifest_path = BENIGN_DIR / "s1_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\n  Manifest saved : {manifest_path}")
print("=" * 55)



S1 — Full Dataset Validation

  deutsch_jozsa — ✓ PASS
    Circuits : 600
    Metadata : 600
    Qubits   : 3–8

  grover — ✓ PASS
    Circuits : 600
    Metadata : 600
    Qubits   : 4–12

  qaoa — ✓ PASS
    Circuits : 600
    Metadata : 600
    Qubits   : 6–14

  vqc — ✓ PASS
    Circuits : 600
    Metadata : 600
    Qubits   : 4–12

  qft — ✓ PASS
    Circuits : 600
    Metadata : 600
    Qubits   : 4–12

  Total circuits : 3000 / 3000 expected
  Duplicate IDs  : 0
  Total errors   : 0

  ✓ ALL CHECKS PASSED — S1 complete

  Manifest saved : C:\Users\zajmal23\OneDrive - University of Oulu and Oamk\quantum_circuits_exp\dataset\circuits\benign\s1_manifest.json
